# 💊 Quantum Drug Discovery with VQE
### Quantum for Healthcare — Quantum for Humanity

This notebook uses the **Variational Quantum Eigensolver (VQE)** to compute the ground-state energy of a molecular Hamiltonian — a foundational step in quantum drug discovery.

**Target diseases:** Tuberculosis (TB), Malaria, Cancer  
**Learning source:** [IBM Quantum Learning — VQE Tutorial](https://learning.quantum.ibm.com/tutorial/variational-quantum-eigensolver)

---

## Why Quantum Drug Discovery?

Classical computers cannot accurately simulate the quantum mechanics of large molecules. For a molecule with $n$ electrons, the Hamiltonian lives in a $2^n$-dimensional Hilbert space — exponentially expensive classically.

**Quantum computers can simulate molecular systems efficiently** — enabling:
- Accurate drug-target binding energy calculations
- Reaction pathway simulation
- Protein-ligand interaction modelling

**VQE** finds the ground-state energy $E_0$ of a molecular Hamiltonian $H$ by minimising:
$$E_0 \leq \langle \psi(\theta) | H | \psi(\theta) \rangle$$

over parameterised quantum circuit states $|\psi(\theta)\rangle$.

In [ ]:
# Install required packages (run once)
# !pip install qiskit qiskit-nature qiskit-algorithms pyscf

import numpy as np
import matplotlib.pyplot as plt

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import Estimator

print('✅ Imports successful')

## Step 1: Define the Molecule

We start with **hydrogen (H₂)** — the simplest molecule with quantum interactions. The same framework applies to drug-relevant molecules (ATP, isoniazid for TB, artemisinin for malaria).

In [ ]:
# H2 molecule at equilibrium bond length
# Geometry: two H atoms separated by 0.735 Angstroms
driver = PySCFDriver(
    atom='H 0 0 0; H 0 0 0.735',
    basis='sto3g'
)

problem = driver.run()
print(f'Molecule: H₂')
print(f'Number of spatial orbitals: {problem.num_spatial_orbitals}')
print(f'Number of particles (alpha, beta): {problem.num_particles}')
print(f'Nuclear repulsion energy: {problem.nuclear_repulsion_energy:.6f} Hartree')

## Step 2: Map to Qubit Hamiltonian

In [ ]:
# Jordan-Wigner mapping: fermions → qubits
mapper = JordanWignerMapper()

# Get second-quantised Hamiltonian
hamiltonian = problem.hamiltonian
second_q_op = hamiltonian.second_q_op()

# Map to qubit operator
qubit_op = mapper.map(second_q_op)
print(f'Number of qubits required: {qubit_op.num_qubits}')
print(f'Number of Pauli terms: {len(qubit_op)}')
print(f'\nHamiltonian (truncated):')
for term in list(qubit_op)[:5]:
    print(f'  {term}')
print('  ...')

## Step 3: Classical Baseline — Full Configuration Interaction (FCI)

In [ ]:
# Exact classical diagonalisation (only feasible for small molecules)
numpy_solver = NumPyMinimumEigensolver()
fci_solver = GroundStateEigensolver(mapper, numpy_solver)
fci_result = fci_solver.solve(problem)

fci_energy = fci_result.total_energies[0]
print(f'FCI Ground State Energy (H₂): {fci_energy:.6f} Hartree')
print(f'  = {fci_energy * 27.211:.4f} eV  (1 Hartree = 27.211 eV)')

## Step 4: Quantum VQE Solution — UCCSD Ansatz

In [ ]:
# Build UCCSD (Unitary Coupled Cluster Singles and Doubles) ansatz
# This is the chemical accuracy ansatz used in quantum chemistry VQE

ansatz = UCCSD(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    mapper=mapper,
    initial_state=HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        mapper=mapper
    )
)

print(f'UCCSD Circuit: {ansatz.num_qubits} qubits, {ansatz.num_parameters} parameters')
ansatz.decompose().draw('mpl', style='clifford')

In [ ]:
# Run VQE
vqe = VQE(
    estimator=Estimator(),
    ansatz=ansatz,
    optimizer=SLSQP(maxiter=300)
)

vqe_solver = GroundStateEigensolver(mapper, vqe)
vqe_result = vqe_solver.solve(problem)

vqe_energy = vqe_result.total_energies[0]
error = abs(vqe_energy - fci_energy) * 1000  # in milli-Hartree

print(f'\n⚛️  VQE Ground State Energy: {vqe_energy:.6f} Hartree')
print(f'📊  FCI Reference:            {fci_energy:.6f} Hartree')
print(f'📏  Error:                    {error:.4f} mHa  (chemical accuracy threshold: 1 mHa)')
if error < 1.0:
    print('✅  Chemical accuracy achieved!')
else:
    print('⚠️  Increase circuit depth (reps) for better accuracy')

## Step 5: Potential Energy Surface — Bond Dissociation Curve

In [ ]:
# Compute VQE energy at different H-H bond lengths
# This reveals the dissociation energy — critical for understanding
# chemical bonds in drug molecules

bond_lengths = np.arange(0.4, 2.5, 0.2)
vqe_energies = []
fci_energies = []

for d in bond_lengths:
    drv = PySCFDriver(atom=f'H 0 0 0; H 0 0 {d:.2f}', basis='sto3g')
    prob = drv.run()
    
    # FCI
    fci_r = GroundStateEigensolver(mapper, NumPyMinimumEigensolver()).solve(prob)
    fci_energies.append(fci_r.total_energies[0])
    
    # VQE
    ans = UCCSD(prob.num_spatial_orbitals, prob.num_particles, mapper,
                initial_state=HartreeFock(prob.num_spatial_orbitals, prob.num_particles, mapper))
    vqe_r = GroundStateEigensolver(mapper, VQE(Estimator(), ans, SLSQP(maxiter=200))).solve(prob)
    vqe_energies.append(vqe_r.total_energies[0])

# Plot
plt.figure(figsize=(8, 5))
plt.plot(bond_lengths, fci_energies, 'o-', color='#8B5CF6', linewidth=2, label='FCI (Exact)')
plt.plot(bond_lengths, vqe_energies, 's--', color='#FF6B9D', linewidth=2, label='VQE (Quantum)')
plt.axvline(0.735, color='gray', linestyle=':', alpha=0.7, label='Equilibrium (0.735 Å)')
plt.xlabel('Bond Length (Å)', fontsize=12)
plt.ylabel('Total Energy (Hartree)', fontsize=12)
plt.title('H₂ Potential Energy Surface\nQuantum VQE vs Classical FCI', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 6: Scaling to Drug-Relevant Molecules

The same VQE framework applies to disease-relevant molecules:

| Molecule | Disease | Atoms | Qubits (approx) | Quantum Advantage |
|---|---|---|---|---|
| Isoniazid (INH) | Tuberculosis | 14 | ~40 | Binding to InhA enzyme |
| Artemisinin | Malaria | 42 | ~120 | Antimalarial mechanism |
| Caffeine | Adenosine receptors | 24 | ~70 | Drug-receptor binding |
| Penicillin | Bacterial infections | 41 | ~115 | β-lactam ring stability |

```python
# Example: Isoniazid fragment (simplified)
driver = PySCFDriver(
    atom='N 0 0 0; N 0 0 1.45; C 0 1.23 0.72',  # pyridine-like fragment
    basis='sto3g'
)
# → Scale up with fault-tolerant IBM Quantum hardware (2027+)
```

## 🌍 Humanitarian Impact

- **TB kills 1.6 million people/year** — quantum simulation can accelerate new drug candidates from 15 years to 3–5 years
- **Malaria affects 240 million/year** — better artemisinin derivatives via quantum molecular design
- **Open-source quantum chemistry** democratises drug discovery for neglected diseases

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*